# ការណែនាំអំពីការរចនាប្រាំ
ការរចនាប្រាំគឺជាការប្រព្រឹត្តទៅនៃការរចនានិងបង្កើតប្រាំសម្រាប់ភារកិច្ចកំណត់ភាសាប្រព័ន្ធធម្មជាតិ។ វាពាក់ព័ន្ធនឹងការជ្រើសរើសប្រាំដែលត្រឹមត្រូវ ការកែសម្រួលប៉ារ៉ាម៉ែត្រ រួមជាមួយការវាយតម្លៃលទ្ធផលរបស់វា។ ការរចនាប្រាំមានសារៈសំខាន់សម្រាប់ការសម្រេចបាននូវភាពត្រឹមត្រូវខ្ពស់ និងប្រសិទ្ធភាពក្នុងម៉ូដែល NLP។ នៅផ្នែកនេះ យើងនឹងសិក្សាពីមូលដ្ឋាននៃការរចនាប្រាំដោយប្រើម៉ូដែល OpenAI សម្រាប់ការត្រៀមសំរាប់សិក្សា។


### ការហ្វឹកហាត់ ១៖ ការចែកពាក្យ
សូមស្ទង់មតិក្នុងការចែកពាក្យដោយប្រើ tiktoken ដែលជាម៉ាស៊ីនចែកពាក្យលឿនប្រភពបើកពី OpenAI
មើល [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) សម្រាប់ឧទាហរណ៍បន្ថែម។


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### ការអនុវត្ត 2: ផ្ទៀងផ្ទាត់ការតំឡើងកូនសោម៉ូដែល Microsoft Foundry

> **កំណត់ចំណាំ:** GitHub Models នឹងបញ្ឈប់ប្រើនៅចុងខែកក្កដា 2026។ ការអនុវត្តនេះប្រើ [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) ផ្ទាល់ដែលផ្តល់ជូននូវកាថ្លុកម៉ូដែលសាកល្បងឥតគិតថ្លៃដូចគ្នា និងបទពិសោធន៍ Azure AI Inference SDK ។

ធ្វើការរត់កូដខាងក្រោមដើម្បីបញ្ជាក់ថាចំណុចបញ្ចូល Microsoft Foundry Models របស់អ្នកត្រូវបានរៀបចំបានត្រឹមត្រូវ។ កូដនេះនឹងព្យាយាមបញ្ចូលបង្ហាញមួយ​ដ៏សាមញ្ញ និងផ្ទៀងផ្ទាត់នូវការសម្រេច។ អត្ថបទបញ្ចូល `oh say can you see` គួរតែបញ្ចប់ជាជួរដូចជា `by the dawn's early light..`


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

# temperature/top_p need a non-reasoning model (gpt-5 rejects them), so use a Llama model here.
model_name = os.environ.get("AZURE_INFERENCE_CHAT_MODEL", "Llama-3.3-70B-Instruct")

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### លំហាត់ 3៖ ការបង្កើតទេ
ស្វែងយល់ពីអ្វីដែលកើតឡើងពេលអ្នកស្នើសុំឲ្យ LLM ត្រឡប់មកវិញនូវការសម្រួលសម្រាប់បរិបទអំពីប្រធានបទដែលប្រហែលជាមិនមាន អោយអ្នកប្រមូលពត៌មានអំពីប្រធានបទដែលវាអាចមិនស្គាល់ព្រោះវា​គ្មានក្នុងឯកសារបណ្ដុះបណ្ដាលរបស់វា (ថ្មីជាងនេះ) មើលថាតើយុទ្ធសាស្ត្រតំណកប្រែប្រួលបែបណាអំឡុងពេលអ្នកព្យាយាមប្រើបរិបទផ្សេង ឬម៉ូឌែលផ្សេង។


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### អនុវត្តន៍ ៤៖ នៅលើមូលដ្ឋានសេចក្តីណែនាំ  
ប្រើអថេរ "text" ដើម្បីកំណត់មាតិកាមូលដ្ឋាន  
និងអថេរ "prompt" ដើម្បីផ្តល់សេចក្តីណែនាំដែលទាក់ទងនឹងមាតិកាមូលដ្ឋាននោះ។  

នៅទីនេះយើងស្នើឲ្យម៉ូដែលសង្ខេបអត្ថបទសម្រាប់សិស្សថ្នាក់ទីពីរ  


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### លំហាត់ ៥: សំណួរលំបាក 
សាកល្បងការស្នើសុំដែលមានសារពីប្រព័ន្ធ អ្នកប្រើ និងជួយ 
ប្រព័ន្ធកំណត់បរិបទជួយ 
សារអ្នកប្រើ និងជួយផ្តល់បរិបទការសន្ទនាច្រើនជំរៅ 

សូមសម្គាល់ពីរបៀបដែលបុគ្គលិកភាពជួយត្រូវបានកំណត់ជា "សម្រួល" នៅក្នុងបរិបទប្រព័ន្ធ។ 
សាកល្បងប្រើបុគ្គលិកភាពផ្សេងពីនេះ។ ឬសាកល្បងជួរដំណើរសារបញ្ចូល/លទ្ធផលផ្សេងទៀត


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### គំនាត់ហាត់ប្រាណ៖ សិក្សាអំពីការយល់ឃើញរបស់អ្នក
ឧទាហរណ៍ខាងលើផ្តល់ឱ្យអ្នកនូវលំនាំដែលអ្នកអាចប្រើបានដើម្បីបង្កើតសំណុំបញ្ជា​ថ្មីៗ (សាមញ្ញ, ស្មុគស្មាញ, សេចក្តីណែនាំ ល។) - សូមព្យាយាមបង្កើតហាត់ប្រាណផ្សេងទៀតដើម្បីសិក្សាអំពីគំនិតផ្សេងទៀតដែលយើងបាននិយាយដូចជា ឧទាហរណ៍, ការផ្តល់សញ្ញា និងផ្សេងៗទៀត។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
